In [1]:
%pip install tensorflow numpy


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import warnings
import tensorflow as tf 
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Flatten, Input
from tensorflow.keras.callbacks import Callback
import numpy as np

# Suppress all Python warnings
warnings.filterwarnings('ignore')

# Set TensorFlow log level to suppress warnings and info messages
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Step 1: Set Up the Environment
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data() 
x_train, x_test = x_train / 255.0, x_test / 255.0 
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)


In [3]:
# Step 2: Define the Model

model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(128, activation='relu'),
    Dense(10)
])


In [4]:
# Step 3: Define Loss Function and Optimizer

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True) 
optimizer = tf.keras.optimizers.Adam()


Implement the Custom Training Loop

In [5]:
# Step 4: Implement the Custom Training Loop

epochs = 2  # Número de vezes que o modelo vai percorrer todo o dataset

# train_dataset = train_dataset.repeat(epochs)  
# (Comentado) Isso faria o dataset repetir automaticamente, mas aqui estamos controlando manualmente com o loop de épocas

train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)
# Cria um dataset do TensorFlow a partir dos dados e divide em batches de 32 amostras

for epoch in range(epochs):  # Loop de épocas (quantas vezes treinar no dataset inteiro)
    print(f'Start of epoch {epoch + 1}')  # Mostra qual época está começando

    for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
        # Loop sobre cada batch do dataset
        # x_batch_train = entradas
        # y_batch_train = respostas corretas

        with tf.GradientTape() as tape:
            # GradientTape observa todas as operações para calcular gradientes depois

            logits = model(x_batch_train, training=True)  
            # Forward pass → modelo faz previsões com base nos dados de entrada

            loss_value = loss_fn(y_batch_train, logits)  
            # Calcula o erro comparando previsão com valor real

        # Compute gradients and update weights
        grads = tape.gradient(loss_value, model.trainable_weights)
        # Calcula os gradientes → quanto cada peso contribuiu para o erro

        optimizer.apply_gradients(zip(grads, model.trainable_weights))
        # Atualiza os pesos do modelo com base nos gradientes (aprendizado)

        # Logging the loss every 200 steps
        if step % 200 == 0:
            print(f'Epoch {epoch + 1} Step {step}: Loss = {loss_value.numpy()}')
            # Mostra o erro atual a cada 200 batches para monitorar o treino

Start of epoch 1
Epoch 1 Step 0: Loss = 2.4399890899658203
Epoch 1 Step 200: Loss = 0.4160134196281433
Epoch 1 Step 400: Loss = 0.1960025429725647
Epoch 1 Step 600: Loss = 0.15448525547981262
Epoch 1 Step 800: Loss = 0.15414363145828247
Epoch 1 Step 1000: Loss = 0.4197186231613159
Epoch 1 Step 1200: Loss = 0.17987114191055298
Epoch 1 Step 1400: Loss = 0.26763731241226196
Epoch 1 Step 1600: Loss = 0.22596535086631775
Epoch 1 Step 1800: Loss = 0.15923327207565308
Start of epoch 2
Epoch 2 Step 0: Loss = 0.07350891083478928
Epoch 2 Step 200: Loss = 0.1398317813873291
Epoch 2 Step 400: Loss = 0.11598801612854004
Epoch 2 Step 600: Loss = 0.04224017262458801
Epoch 2 Step 800: Loss = 0.0705665647983551
Epoch 2 Step 1000: Loss = 0.26835471391677856
Epoch 2 Step 1200: Loss = 0.06836973875761032
Epoch 2 Step 1400: Loss = 0.15483458340168
Epoch 2 Step 1600: Loss = 0.16366027295589447
Epoch 2 Step 1800: Loss = 0.07885631173849106


Adding Accuracy Metric

In [6]:
import tensorflow as tf  # Importa o TensorFlow (biblioteca principal de Deep Learning)

from tensorflow.keras.models import Sequential  # Modelo sequencial (camadas em ordem)
from tensorflow.keras.layers import Dense, Flatten  # Camadas da rede neural

# Step 1: Set Up the Environment

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
# Carrega o dataset MNIST (imagens de números de 0 a 9)
# x_train → imagens de treino
# y_train → rótulos (número correto)
# x_test → imagens de teste
# y_test → rótulos de teste

# Normalize the pixel values to be between 0 and 1
x_train, x_test = x_train / 255.0, x_test / 255.0 
# Normaliza os pixels (0–255 → 0–1)
# Isso ajuda o modelo a aprender melhor e mais rápido

# Create a batched dataset for training
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)
# Cria um dataset do TensorFlow a partir dos dados
# Divide em batches de 32 exemplos
# Cada batch será usado em uma etapa do treinamento

In [7]:
# Step 2: Define the Model

model = Sequential([ 
    Flatten(input_shape=(28, 28)),  # Flatten the input to a 1D vector
    Dense(128, activation='relu'),  # First hidden layer with 128 neurons and ReLU activation
    Dense(10)  # Output layer with 10 neurons for the 10 classes (digits 0-9)
])


In [8]:
# Step 3: Define Loss Function, Optimizer, and Metric

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)  # Loss function for multi-class classification
optimizer = tf.keras.optimizers.Adam()  # Adam optimizer for efficient training
accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy()  # Metric to track accuracy during training


In [9]:
# Step 4: Implement the Custom Training Loop with Accuracy

epochs = 5  # Número de épocas (quantas vezes o modelo verá todo o dataset)

for epoch in range(epochs):
    print(f'Start of epoch {epoch + 1}')  # Indica o início da época
    
    for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
        # Loop sobre os batches (lotes de dados)

        with tf.GradientTape() as tape:
            # Inicia o rastreamento das operações para calcular gradientes

            # Forward pass: Compute predictions
            logits = model(x_batch_train, training=True)
            # O modelo faz previsões com base nos dados de entrada

            # Compute loss
            loss_value = loss_fn(y_batch_train, logits)
            # Calcula o erro comparando previsão vs valor real
        
        # Compute gradients
        grads = tape.gradient(loss_value, model.trainable_weights)
        # Calcula quanto cada peso contribuiu para o erro (gradientes)

        # Apply gradients to update model weights
        optimizer.apply_gradients(zip(grads, model.trainable_weights))
        # Atualiza os pesos para reduzir o erro
        
        # Update the accuracy metric
        accuracy_metric.update_state(y_batch_train, logits)
        # Atualiza a métrica de acurácia com os dados atuais

        # Log the loss and accuracy every 200 steps
        if step % 200 == 0:
            print(f'Epoch {epoch + 1} Step {step}: Loss = {loss_value.numpy()} Accuracy = {accuracy_metric.result().numpy()}')
            # Mostra o erro e a acurácia atual
    
    # Reset the metric at the end of each epoch
    accuracy_metric.reset_state()
    # Reseta a métrica para começar limpa na próxima época

Start of epoch 1
Epoch 1 Step 0: Loss = 2.3566367626190186 Accuracy = 0.09375
Epoch 1 Step 200: Loss = 0.3674215078353882 Accuracy = 0.8300684094429016
Epoch 1 Step 400: Loss = 0.17946842312812805 Accuracy = 0.8639339208602905
Epoch 1 Step 600: Loss = 0.17953108251094818 Accuracy = 0.8803556561470032
Epoch 1 Step 800: Loss = 0.1546163260936737 Accuracy = 0.8935705423355103
Epoch 1 Step 1000: Loss = 0.4128981828689575 Accuracy = 0.901629626750946
Epoch 1 Step 1200: Loss = 0.17234690487384796 Accuracy = 0.9089040160179138
Epoch 1 Step 1400: Loss = 0.270957350730896 Accuracy = 0.9141907691955566
Epoch 1 Step 1600: Loss = 0.2271317094564438 Accuracy = 0.9174343943595886
Epoch 1 Step 1800: Loss = 0.17732709646224976 Accuracy = 0.9215019345283508
Start of epoch 2
Epoch 2 Step 0: Loss = 0.0876205638051033 Accuracy = 0.96875
Epoch 2 Step 200: Loss = 0.13128694891929626 Accuracy = 0.9637748599052429
Epoch 2 Step 400: Loss = 0.12284690141677856 Accuracy = 0.9604893922805786
Epoch 2 Step 600: Los

Custom Callback for Advanced Logging

In [10]:
import tensorflow as tf  # Importa a biblioteca principal para Deep Learning

from tensorflow.keras.models import Sequential  # Modelo sequencial (camadas empilhadas)
from tensorflow.keras.layers import Dense, Flatten  # Camadas da rede neural

# Step 1: Set Up the Environment

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
# Carrega o dataset MNIST (imagens de dígitos de 0 a 9)
# x_train → imagens de treino (60000 exemplos)
# y_train → rótulos (número correto de cada imagem)
# x_test → imagens de teste
# y_test → rótulos de teste

# Normalize the pixel values to be between 0 and 1
x_train, x_test = x_train / 255.0, x_test / 255.0 
# Normaliza os pixels (antes: 0–255, depois: 0–1)
# Isso melhora a estabilidade do treinamento e facilita o aprendizado

# Create a batched dataset for training
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)
# Converte os dados em um Dataset do TensorFlow (pipeline eficiente)
# Agrupa os dados em batches de 32 amostras

In [11]:
# Step 2: Define the Model

model = Sequential([
    Flatten(input_shape=(28, 28)),  # Flatten the input to a 1D vector
    Dense(128, activation='relu'),  # First hidden layer with 128 neurons and ReLU activation
    Dense(10)  # Output layer with 10 neurons for the 10 classes (digits 0-9)
])


In [12]:
# Step 3: Define Loss Function, Optimizer, and Metric

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)  # Loss function for multi-class classification
optimizer = tf.keras.optimizers.Adam()  # Adam optimizer for efficient training
accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy()  # Metric to track accuracy during training


In [13]:
from tensorflow.keras.callbacks import Callback

# Step 4: Implement the Custom Callback 
class CustomCallback(Callback):
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        print(f'End of epoch {epoch + 1}, loss: {logs.get("loss")}, accuracy: {logs.get("accuracy")}')


In [14]:
# Step 5: Implement the Custom Training Loop with Custom Callback

epochs = 2
custom_callback = CustomCallback()  # Initialize the custom callback

for epoch in range(epochs):
    print(f'Start of epoch {epoch + 1}')
    
    for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
        with tf.GradientTape() as tape:
            # Forward pass: Compute predictions
            logits = model(x_batch_train, training=True)
            # Compute loss
            loss_value = loss_fn(y_batch_train, logits)
        
        # Compute gradients
        grads = tape.gradient(loss_value, model.trainable_weights)
        # Apply gradients to update model weights
        optimizer.apply_gradients(zip(grads, model.trainable_weights))
        
        # Update the accuracy metric
        accuracy_metric.update_state(y_batch_train, logits)

        # Log the loss and accuracy every 200 steps
        if step % 200 == 0:
            print(f'Epoch {epoch + 1} Step {step}: Loss = {loss_value.numpy()} Accuracy = {accuracy_metric.result().numpy()}')
    
    # Call the custom callback at the end of each epoch
    custom_callback.on_epoch_end(epoch, logs={'loss': loss_value.numpy(), 'accuracy': accuracy_metric.result().numpy()})
    
    # Reset the metric at the end of each epoch
    accuracy_metric.reset_state()  # Use reset_state() instead of reset_states()


Start of epoch 1
Epoch 1 Step 0: Loss = 2.374549150466919 Accuracy = 0.09375
Epoch 1 Step 200: Loss = 0.40444856882095337 Accuracy = 0.8320895433425903
Epoch 1 Step 400: Loss = 0.1962769329547882 Accuracy = 0.8667393922805786
Epoch 1 Step 600: Loss = 0.1612100601196289 Accuracy = 0.882903516292572
Epoch 1 Step 800: Loss = 0.14562444388866425 Accuracy = 0.8956382870674133
Epoch 1 Step 1000: Loss = 0.422974169254303 Accuracy = 0.9033466577529907
Epoch 1 Step 1200: Loss = 0.16220739483833313 Accuracy = 0.909736692905426
Epoch 1 Step 1400: Loss = 0.2792624831199646 Accuracy = 0.9145922660827637
Epoch 1 Step 1600: Loss = 0.20039844512939453 Accuracy = 0.9176686406135559
Epoch 1 Step 1800: Loss = 0.1884305477142334 Accuracy = 0.9216927886009216
End of epoch 1, loss: 0.04751771688461304, accuracy: 0.9236833453178406
Start of epoch 2
Epoch 2 Step 0: Loss = 0.09245625883340836 Accuracy = 1.0
Epoch 2 Step 200: Loss = 0.1936396211385727 Accuracy = 0.9601989984512329
Epoch 2 Step 400: Loss = 0.118

In [15]:
from tensorflow.keras.layers import Input, Dense

# Define the input layer
input_layer = Input(shape=(28, 28))  # Input layer with shape (28, 28)

# Flatten the 2D images into 1D vectors before Dense layers
flatten = Flatten()(input_layer)

# Define hidden layers
hidden_layer1 = Dense(64, activation='relu')(flatten)  # First hidden layer with 64 neurons and ReLU activation
hidden_layer2 = Dense(64, activation='relu')(hidden_layer1)  # Second hidden layer with 64 neurons and ReLU activation


In [16]:
output_layer = Dense(10, activation='softmax')(hidden_layer2)

In [17]:
model = Model(inputs=input_layer, outputs=output_layer)

In [18]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # Correct for integer labels 0-9
    metrics=['accuracy']
)

In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
import numpy as np

# Load and preprocess MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0  # Normalize pixel values to [0, 1]

# Train the model
history = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=32
)

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9184 - loss: 0.2791
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9618 - loss: 0.1265  
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9717 - loss: 0.0927
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9771 - loss: 0.0719
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9817 - loss: 0.0584


In [20]:
# Example test data (in practice, use real dataset)
loss, accuracy = model.evaluate(x_test, y_test)

print(f'Test loss:     {loss:.4f}')
print(f'Test accuracy: {accuracy:.4f}')



313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 607us/step - accuracy: 0.9749 - loss: 0.0774
Test loss:     0.0774
Test accuracy: 0.9749
